In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel("Global_risk_Calculation.xlsx")

In [ ]:
df.columns

Index(['GlobalEventID', 'EventDate', 'Actor1CountryCode', 'Actor2CountryCode',
       'EventCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale', 'AvgTone',
       'NumMentions', 'NumSources', 'NumArticles', 'ActionGeo_CountryCode',
       'ActionGeo_Lat', 'ActionGeo_Long'],
      dtype='object')

In [ ]:
# Inserting Columns:


In [ ]:
df['Conflict'] = None
df['GoldStienRisk'] = None
df['ToneRisk'] = None
df['Importance'] = None
display(df.head())

,GlobalEventID,EventDate,Actor1CountryCode,Actor2CountryCode,EventCode,EventRootCode,QuadClass,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_Lat,ActionGeo_Long,Conflict,GoldStienRisk,ToneRisk,Importance
0,1309006226,2025-06-15,UNKNOWN,USA,42,4,1,1.9,1.162791,5,1,5,US,21.1098,-157.5310,None,None,None,None
1,1309006227,2025-06-15,USA,UNKNOWN,43,4,1,2.8,1.162791,5,1,5,US,21.1098,-157.5310,None,None,None,None
2,1309006228,2026-05-16,UNKNOWN,USA,42,4,1,1.9,-4.231434,1,1,1,US,38.8951,-77.0364,None,None,None,None
3,1309006229,2026-05-16,USA,UNKNOWN,43,4,1,2.8,-4.231434,1,1,1,US,38.8951,-77.0364,None,None,None,None
4,1309006230,2026-06-08,FRA,USA,46,4,1,7.0,3.137279,726,121,726,FR,46.0000,2.0000,None,None,None,None


In [ ]:
import numpy as np

# importing Conflict
conditions = [
    df['QuadClass'] == 4,
    df['QuadClass'] == 3,
    df['QuadClass'] == 2,
    df['QuadClass'] == 1
]
choices = [100, 50, 0, 0]
df['Conflict'] = np.select(conditions, choices, default=df['Conflict']) # Use default=df['Conflict'] to keep existing None values if no condition is met.

In [ ]:
import numpy as np
df['GoldStienRisk'] = np.maximum(0, -df['GoldsteinScale'])

In [ ]:
# Normalize GoldStienRisk
df['GoldStienRisk'] = ((df['GoldStienRisk'] - df['GoldStienRisk'].min()) / (df['GoldStienRisk'].max() - df['GoldStienRisk'].min()))*100

In [ ]:
df['ToneRisk'] = np.maximum(0, -df['AvgTone'])

In [ ]:
#Normalize ToneRisk
df['ToneRisk'] = ((df['ToneRisk'] - df['ToneRisk'].min()) / (df['ToneRisk'].max() - df['ToneRisk'].min()))*100

In [ ]:
#Importance = 0.5 * NumMentions + 0.3 * NumSources + 0.2 * NumArticles
df['Importance'] = 0.5 * df['NumMentions'] + 0.3 * df['NumSources'] + 0.2 * df['NumArticles']

In [ ]:
# Normalize Importance
df['Importance'] = ((df['Importance'] - df['Importance'].min()) / (df['Importance'].max() - df['Importance'].min()))*100

In [ ]:
# Now we need to find out ConflictScore
df['Conflict_Score'] = 0.4*df['Conflict'] + 0.3*df['GoldStienRisk'] + 0.2*df['ToneRisk'] + 0.1*df['Importance']

In [ ]:
df['MediaSentimentRisk'] = df['ToneRisk']

In [ ]:
conditions = [
    df['QuadClass'] == 4,
    df['QuadClass'] == 3,
    df['QuadClass'] == 2,
    df['QuadClass'] == 1
]
choices = [100, 50, 0, 0]
df['MilitaryActivity'] = np.select(conditions, choices)

In [ ]:
# RootCodeServerity Score:
severity_map = {
    1: 5, 2: 10, 3: 15, 4: 20,
    5: 10, 6: 15, 7: 20, 8: 15,
    9: 25, 10: 40, 11: 50, 12: 55,
    13: 70, 14: 75, 15: 80, 16: 85,
    17: 90, 18: 95, 19: 100, 20: 100
}

df["RootCodeSeverity"] = df["EventRootCode"].map(severity_map)


In [ ]:
# Severity Score = 0.4* RootCodeSeverity + 0.3*(-GoldstienScale) + 0.2*(-AvgTone) + 0.1 * Normal numMentions
df['NumMentions'] = ((df['NumMentions'] - df['NumMentions'].min()) / (df['NumMentions'].max() - df['NumMentions'].min()))*100
df['Severity_Score'] = 0.4*df['RootCodeSeverity'] + 0.3*df['GoldStienRisk'] + 0.2*df['ToneRisk'] + 0.1*df['NumMentions']

In [ ]:
# adjusted Severity
df['Adjusted_Severity_Score'] = df['Severity_Score']*(1+(df['ToneRisk']/10))

In [ ]:
# SupplyChainScore =0.35 × Severity+ 0.25 × GoldsteinImpact+0.20 × MediaAttention+0.10 × NumSources+0.10 × ToneRisk
df['SupplyChainScore'] = 0.35*df['Adjusted_Severity_Score'] + 0.25*df['GoldStienRisk'] + 0.2*df['MediaSentimentRisk'] + 0.1*df['NumSources'] + 0.10*df['ToneRisk']

In [ ]:
df['PoliticalScore'] = (
    0.40*df['Adjusted_Severity_Score'] +
    0.25*df['ToneRisk'] +
    0.20*df['GoldStienRisk'] +
    0.15*df['NumMentions']
)

In [ ]:
import numpy as np
df['SanctionScore'] = np.where(df['EventRootCode'].isin([10, 12, 13, 16, 17]), df['Adjusted_Severity_Score'], 0)

In [ ]:
df

,GlobalEventID,EventDate,Actor1CountryCode,Actor2CountryCode,EventCode,EventRootCode,QuadClass,GoldsteinScale,AvgTone,NumMentions,...,Conflict_Score,MediaSentimentRisk,MilitaryActivity,RootCodeSeverity,Severity_Score,Adjusted_Severity_Score,SupplyChainScore,PoliticalScore,SanctionEvent,SanctionScore
0,1309006226,2025-06-15,UNKNOWN,USA,42,4,1,1.9,1.162791,0.093480,...,0.00934,0.000000,0,20,8.009348,8.009348,2.903272,3.217761,False,0.0
1,1309006227,2025-06-15,USA,UNKNOWN,43,4,1,2.8,1.162791,0.093480,...,0.00934,0.000000,0,20,8.009348,8.009348,2.903272,3.217761,False,0.0
2,1309006228,2026-05-16,UNKNOWN,USA,42,4,1,1.9,-4.231434,0.000000,...,5.793809,28.969045,0,20,13.793809,53.753156,27.604318,28.743524,False,0.0
3,1309006229,2026-05-16,USA,UNKNOWN,43,4,1,2.8,-4.231434,0.000000,...,5.793809,28.969045,0,20,13.793809,53.753156,27.604318,28.743524,False,0.0
4,1309006230,2026-06-08,FRA,USA,46,4,1,7.0,3.137279,16.943211,...,1.813057,0.000000,0,20,9.694321,9.694321,15.493012,6.419210,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1309009029,2026-06-15,IRN,UNKNOWN,30,3,1,4.0,-0.760248,0.023370,...,1.044291,5.204776,0,15,7.043292,10.709168,5.509642,5.588367,False,0.0
1996,1309009030,2026-06-15,IRN,CHE,30,3,1,4.0,0.328407,0.116850,...,0.011676,0.000000,0,15,6.011685,6.011685,2.204090,2.422201,False,0.0
1997,1309009031,2026-06-15,IRN,CHE,30,3,1,4.0,0.328407,0.537509,...,0.053708,0.000000,0,15,6.053751,6.053751,2.218813,2.502127,False,0.0
1998,1309009032,2026-06-15,IRN,UNKNOWN,30,3,1,4.0,2.895753,0.116850,...,0.011676,0.000000,0,15,6.011685,6.011685,2.204090,2.422201,False,0.0


In [ ]:
df.columns

Index(['GlobalEventID', 'EventDate', 'Actor1CountryCode', 'Actor2CountryCode',
       'EventCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale', 'AvgTone',
       'NumMentions', 'NumSources', 'NumArticles', 'ActionGeo_CountryCode',
       'ActionGeo_Lat', 'ActionGeo_Long', 'Conflict', 'GoldStienRisk',
       'ToneRisk', 'Importance', 'Conflict_Score', 'MediaSentimentRisk',
       'MilitaryActivity', 'RootCodeSeverity', 'Severity_Score',
       'Adjusted_Severity_Score', 'SupplyChainScore', 'PoliticalScore',
       'SanctionEvent', 'SanctionScore'],
      dtype='object')

In [ ]:
# Now We are able to Calculate GLobal Risk
df['Global Risk'] = ( 0.3 * df['Conflict_Score']
                     + 0.2 * df['GoldStienRisk']
                      + 0.15 * df['MediaSentimentRisk']
                      + 0.10 * df['MilitaryActivity']
                      + 0.10 * df['SupplyChainScore']
                      + 0.05 * df['PoliticalScore']
                      + 0.10* df['SanctionScore'] )

In [ ]:
df['Global Risk'] = ((df['Global Risk'] - df['Global Risk'].min())/(df['Global Risk'].max() - df['Global Risk'].min()))*100

In [ ]:
df['Global Risk'].min()

0.0

In [ ]:
# Fetching out those three column ActionGeo_Lat , ActionGeo_Long , Global Risk
Data_required = df[['ActionGeo_Lat', 'ActionGeo_Long', 'Global Risk']]
display(Data_required.head())

,ActionGeo_Lat,ActionGeo_Long,Global Risk
0,21.1098,-157.5310,0.18105
1,21.1098,-157.5310,0.18105
2,38.8951,-77.0364,5.507711
3,38.8951,-77.0364,5.507711
4,46.0000,2.0000,1.243533


In [ ]:
# Downloading the Data Excel File


In [ ]:
output_filename = 'Data_required.xlsx'
df.to_excel(output_filename, index=False)
print(f"DataFrame saved to '{output_filename}'")

DataFrame saved to 'Data_required.xlsx'


You can download the processed Excel file [here](Data_required.xlsx).

In [ ]:
target_lat = 39.828175
target_long = -98.5795

# Find entries that match the given latitude and longitude
matching_risk = Data_required[(abs(Data_required['ActionGeo_Lat'] - target_lat) < 0.001) & (abs(Data_required['ActionGeo_Long'] - target_long) < 0.001)]

if not matching_risk.empty:
    print(f"Global Risk at Latitude {target_lat}° N, Longitude {target_long}° E:")
    display(matching_risk[['Global Risk']]) # Changed .averages() to .mean()
else:
    print(f"No exact match found for Latitude {target_lat}° N, Longitude {target_long}° E. You may want to check nearby locations or consider a broader search criteria.")

Global Risk at Latitude 39.828175° N, Longitude -98.5795° E:


,Global Risk
28,25.933003
87,3.247721
126,0.179417
138,0.760904
148,24.034785
...,...
1812,0.064524
1969,0.237831
1972,1.452227
1975,1.360089


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 68.3 MB/s eta 0:00:00


In [ ]:

import streamlit as st
import pandas as pd

from risk_engine import calculate_global_risk

st.set_page_config(
    page_title="Global Supply Chain Risk",
    layout="wide"
)

st.title("🌍 AI Global Supply Chain Risk Dashboard")

uploaded_file = st.file_uploader(
    "Global_risk_Calculation.xlsx",
    key="file_uploader", # Added missing comma here
    type=["excel"]
)

# Only proceed if a file has been uploaded
if uploaded_file is not None:
    df = pd.read_excel(uploaded_file)

    st.success("File Uploaded Successfully")
    if st.button("Calculate Global Risk"):

            result = calculate_global_risk(df)

            st.subheader("Calculated Dataset")

            st.dataframe(result)

            csv = result.to_csv(index=False)

            st.download_button(
                "Download Result",
                csv,
                "Global_Risk.csv",
                "text/csv"
            )

            st.metric(
                "Average Global Risk",
                round(result["Global Risk"].mean(),2)
            )

            st.metric(
                "Maximum Risk",
                round(result["Global Risk"].max(),2)
            )

            st.metric(
                "Countries",
                result["ActionGeo_CountryCode"].nunique()
            )

            st.bar_chart(
                result["Global Risk"]
            )
else:
    st.info("Please upload an Excel file to proceed.")

2026-07-03 08:56:47.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.961 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.962 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.965 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.966 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.967 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.968 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-03 08:56:47.969 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
%%writefile risk_engine.py
import pandas as pd
import numpy as np

def calculate_global_risk(df):
    # Re-apply all calculation steps to the input DataFrame

    # GoldStienRisk
    df['GoldStienRisk'] = np.maximum(0, -df['GoldsteinScale'])
    df['GoldStienRisk'] = ((df['GoldStienRisk'] - df['GoldStienRisk'].min()) / (df['GoldStienRisk'].max() - df['GoldStienRisk'].min()))*100

    # ToneRisk
    df['ToneRisk'] = np.maximum(0, -df['AvgTone'])
    df['ToneRisk'] = ((df['ToneRisk'] - df['ToneRisk'].min()) / (df['ToneRisk'].max() - df['ToneRisk'].min()))*100

    # Importance
    df['Importance'] = 0.5 * df['NumMentions'] + 0.3 * df['NumSources'] + 0.2 * df['NumArticles']
    df['Importance'] = ((df['Importance'] - df['Importance'].min()) / (df['Importance'].max() - df['Importance'].min()))*100

    # Conflict
    conditions_conflict = [
        df['QuadClass'] == 4,
        df['QuadClass'] == 3,
        df['QuadClass'] == 2,
        df['QuadClass'] == 1
    ]
    choices_conflict = [100, 50, 0, 0]
    df['Conflict'] = np.select(conditions_conflict, choices_conflict)

    # Conflict_Score
    df['Conflict_Score'] = 0.4*df['Conflict'] + 0.3*df['GoldStienRisk'] + 0.2*df['ToneRisk'] + 0.1*df['Importance']

    # MediaSentimentRisk
    df['MediaSentimentRisk'] = df['ToneRisk']

    # MilitaryActivity
    conditions_military = [
        df['QuadClass'] == 4,
        df['QuadClass'] == 3,
        df['QuadClass'] == 2,
        df['QuadClass'] == 1
    ]
    choices_military = [100, 50, 0, 0]
    df['MilitaryActivity'] = np.select(conditions_military, choices_military)

    # RootCodeSeverity
    severity_map = {
        1: 5, 2: 10, 3: 15, 4: 20,
        5: 10, 6: 15, 7: 20, 8: 15,
        9: 25, 10: 40, 11: 50, 12: 55,
        13: 70, 14: 75, 15: 80, 16: 85,
        17: 90, 18: 95, 19: 100, 20: 100
    }
    df["RootCodeSeverity"] = df["EventRootCode"].map(severity_map)

    # NumMentions (Normalized for Severity Score)
    df['NumMentions_Normalized'] = ((df['NumMentions'] - df['NumMentions'].min()) / (df['NumMentions'].max() - df['NumMentions'].min()))*100

    # Severity_Score
    df['Severity_Score'] = 0.4*df['RootCodeSeverity'] + 0.3*df['GoldStienRisk'] + 0.2*df['ToneRisk'] + 0.1*df['NumMentions_Normalized']

    # Adjusted_Severity_Score
    df['Adjusted_Severity_Score'] = df['Severity_Score']*(1+(df['ToneRisk']/10))

    # SupplyChainScore
    df['SupplyChainScore'] = 0.35*df['Adjusted_Severity_Score'] + 0.25*df['GoldStienRisk'] + 0.2*df['MediaSentimentRisk'] + 0.1*df['NumSources'] + 0.10*df['ToneRisk']

    # PoliticalScore
    df['PoliticalScore'] = (
        0.40*df['Adjusted_Severity_Score'] +
        0.25*df['ToneRisk'] +
        0.20*df['GoldStienRisk'] +
        0.15*df['NumMentions_Normalized']
    )

    # SanctionScore
    df['SanctionScore'] = np.where(df['EventRootCode'].isin([10, 12, 13, 16, 17]), df['Adjusted_Severity_Score'], 0)

    # Global Risk
    df['Global Risk'] = ( 0.3 * df['Conflict_Score']
                         + 0.2 * df['GoldStienRisk']
                          + 0.15 * df['MediaSentimentRisk']
                          + 0.10 * df['MilitaryActivity']
                          + 0.10 * df['SupplyChainScore']
                          + 0.05 * df['PoliticalScore']
                          + 0.10* df['SanctionScore'] )

    df['Global Risk'] = ((df['Global Risk'] - df['Global Risk'].min())/(df['Global Risk'].max() - df['Global Risk'].min()))*100

    return df


Writing risk_engine.py


In [ ]:
# These metrics and charts have been moved to the main Streamlit application cell (l_FrgclEJw1F) to ensure 'result' is defined.